# Peek at the incoming data

Real messages from the landing zone, heartbeats skipped: ET journeys, SX situations and vehicle
positions, printed readably and as pandas tables to poke at. Read-only; every run of a cell
samples different messages from the hour being written.

Uses `peek.py` from the deployed collector (`~/dat535-collector`), or from this checkout when
run elsewhere. Commit this notebook with outputs cleared: they contain real data.

In [ ]:
import sys
from pathlib import Path

for p in (Path.home() / "dat535-collector", Path.cwd().parent / "collector"):
    if (p / "peek.py").exists():
        sys.path.insert(0, str(p))
        break
import peek

import pandas as pd
pd.set_option("display.max_columns", 50, "display.max_colwidth", 80)

LANDING = "/data/dat535/landing"
N = 2          # messages per sample
# None = the hour being written (topped up from the last closed hours); or a closed hour,
# e.g. f"{LANDING}/siri-et/date=2026-10-17/hour=12.jsonl.gz" (reads its first messages)
FILES = {"et": None, "sx": None, "vehicles": None}


def show(feed, msgs):
    for m in msgs:
        header, body = peek.format_message(feed, m)
        print(f"── {header}\n{body}\n")

## ET: estimated timetable

One push carries one or more journeys (`EstimatedVehicleJourney`). Calls the vehicle has passed
are `RecordedCall` (aimed vs. actual times), the rest are `EstimatedCall` (aimed vs. expected).

In [ ]:
et = peek.sample("et", N, FILES["et"], LANDING)
show("et", et)

In [ ]:
# The first journey above as tables: journey fields, then one row per call
journey = et[0]["items"][0]
display(pd.Series(peek.flatten(journey), name="value").to_frame())
pd.DataFrame(peek.calls(journey))

## SX: situations (alerts)

Mostly heartbeats. Early in an hour there may be no data yet; the sample then comes from the
last closed hours.

In [ ]:
sx = peek.sample("sx", N, FILES["sx"], LANDING)
show("sx", sx)

In [ ]:
pd.DataFrame([peek.flatten(s) for m in sx for s in m["items"]])

## Vehicle positions

Each message holds up to 20 vehicle updates (GraphQL subscription, batched by Entur).

In [ ]:
veh = peek.sample("vehicles", N, FILES["vehicles"], LANDING)
show("vehicles", veh)

In [ ]:
# A bigger sample as one table, one row per vehicle update
vehicles = pd.json_normalize([v for m in peek.sample("vehicles", 200, landing=LANDING) for v in m["items"]])
print(len(vehicles), "updates from", vehicles.vehicleId.nunique(), "vehicles")
vehicles[["lastUpdated", "codespace.codespaceId", "mode", "line.publicCode", "line.lineName",
          "delay", "location.latitude", "location.longitude", "monitoredCall.stopPointRef",
          "occupancyStatus"]].head(20)

In [ ]:
# Delay (seconds) per codespace in this sample
vehicles.groupby("codespace.codespaceId").delay.describe().sort_values("count", ascending=False)